# 01 - Chargement et preparation des images

Verification d'integrite, exploration, split stratife 70/15/15, augmentation reservee a l'entrainement et chargement par lots. Les images sources ne sont pas deplacees.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import (
    creer_datasets_images,
    inspecter_dataset_images,
    repartir_images,
)

DATA_PATH = PROJECT_ROOT / "data" / "raw"
print(f"Dataset : {DATA_PATH}")

In [ ]:
images, fichiers_invalides = inspecter_dataset_images(DATA_PATH)
print(f"Images valides : {len(images)}")
print(f"Fichiers ignores (extension invalide ou image corrompue) : {len(fichiers_invalides)}")
if not fichiers_invalides.empty:
    display(fichiers_invalides.head(10))
display(images["label"].value_counts().rename_axis("Classe").to_frame("Images valides"))

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=images, x="label", order=sorted(images["label"].unique()))
plt.title("Nombre d'images valides par classe")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, images["label"].nunique(), figsize=(4 * images["label"].nunique(), 4))
for ax, (classe, groupe) in zip(axes.flat if hasattr(axes, "flat") else [axes], images.groupby("label", sort=True)):
    with Image.open(groupe.iloc[0]["path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(classe)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
repartitions = repartir_images(images, taille_test=0.15, taille_validation=0.15, seed=42)
effectifs = {nom: len(table) for nom, table in repartitions.items()}
print("Effectifs avant augmentation :")
for nom, effectif in effectifs.items():
    print(f"{nom.capitalize():12s}: {effectif} images ({effectif / len(images):.1%})")

datasets, split_workspace, class_names, effectifs_classes = creer_datasets_images(
    repartitions, taille_image=224, taille_lot=32, seed=42
)
train_ds = datasets["train"]
validation_ds = datasets["validation"]
test_ds = datasets["test"]
nb_images_train_equilibre = max(repartitions["train"]["label"].value_counts()) * len(class_names)
print(f"Classes (ordre des etiquettes) : {class_names}")
print(f"Train apres reequilibrage par sur-echantillonnage : {nb_images_train_equilibre} exemples par epoch")
print("Normalisation InceptionV3 : pixels float32 dans [-1, 1].")
print("Augmentations train uniquement : retournement horizontal, flou gaussien et bruit aleatoire.")